[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

i.e.
micro-batch 1 → compute gradient 
micro-batch 2 → add gradient 
micro-batch 3 → add gradient 
micro-batch 4 → add gradient
then update model once(step)

instead of updating every batch, we use this when we cant set the batch size too large but still want large-batch gradient

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [70]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    accumulation_steps = len(micro_batches)
    optimizer.zero_grad()
    for micro_batch in micro_batches:
        loss = loss_fn(model(micro_batch[0]), micro_batch[-1]) / accumulation_steps  # scale loss by micro-batch size
        print(loss.item())
        loss.backward()
    optimizer.step()
    pass  # zero_grad, loop (forward, scale loss, backward), step
    return loss.item()  # return the final loss value for testing

In [72]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

0.16650864481925964
0.22486428916454315
0.24887478351593018
1.2565451860427856
Loss: 1.2565451860427856


In [74]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
0.8133244514465332
0.3862258195877075
  ✅ [1/3] Matches full batch update (1.8ms)
2.477104663848877
  ✅ [2/3] Returns loss value (0.4ms)
4.173949718475342
  ✅ [3/3] Parameters actually update (0.3ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (2.4ms total)
  Progress saved. Run status() to see your dashboard.

